# Structured Output and Model-Agnostic Design

Structured output asks a model to return data that follows a schema so application code can validate and use it safely.


## What to learn

- A schema defines required fields, types, allowed values, and constraints.
- Provider enforcement improves syntax but does not guarantee factual correctness.
- Application validation catches semantic and business-rule errors.
- A provider adapter keeps model-specific request and response details out of business logic.

## Decision rule

Use the smallest schema that represents the decision. Validate after generation, retry only repairable failures, cap retries, preserve raw responses for debugging, and test adapters with the same contract suite.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# ALTERNATIVE FRAMEWORK EXAMPLE: Pydantic AI validated output
# Import the dependencies used by this example.
import os
from pydantic import BaseModel, Field

# Define the data shape and small operations before running them.
class SupportDecision(BaseModel):
    category: str = Field(description="billing, technical, or general")
    urgency: int = Field(ge=1, le=5)
    needs_human: bool

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to run this example.")
else:
    from pydantic_ai import Agent

    # Configure the framework object; this line prepares it but may not execute it yet.
    agent = Agent(
        "openai:gpt-5.6-sol",
        output_type=SupportDecision,
        instructions="Classify the request. Escalate security or payment-risk cases.",
    )
    result = await agent.run("I was charged twice and do not recognize one payment.")

    # Pydantic has already checked the field names, types, and urgency range.
    decision = result.output
    print(decision.model_dump())


In [ ]:
"""Validate-and-retry + task-tier routing, in miniature.

Shows the two patterns without external dependencies. In production the
validator is Pydantic/Zod and the router is LiteLLM/AI SDK config.
"""
# Import the dependencies used by this example.
import json

MODEL_CONFIG = {
    # task -> tier; tier -> current model. A swap = edit this dict only.
    "tiers": {"cheap-fast": "haiku-4.5", "frontier": "fable-5"},
    "tasks": {"classify_response": "cheap-fast", "synthesize_themes": "frontier"},
}


# Define the data shape and small operations before running them.
def route(task: str) -> str:
    return MODEL_CONFIG["tiers"][MODEL_CONFIG["tasks"][task]]


def validate(raw: str) -> tuple[dict | None, str | None]:
    """Schema + business-rule validation. Strict mode guarantees parse,
    but NOT these semantic rules — that's why this layer survives."""
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        return None, f"invalid JSON: {e}"
    allowed = {"pricing", "onboarding", "support", "other"}
    if data.get("theme") not in allowed:
        return None, f"theme must be one of {sorted(allowed)}"
    if not 0.0 <= data.get("confidence", -1) <= 1.0:
        return None, "confidence must be in [0, 1]"
    return data, None


def call_with_retry(prompt: str, model: str, max_retries: int = 2) -> dict:
    """Re-ask pattern: feed the validation error back to the model."""
    attempts = [  # simulated model outputs: bad enum, then valid
        '{"theme": "cost", "confidence": 0.9}',
        '{"theme": "pricing", "confidence": 0.9}',
    ]
    # Simulated: a real client would send the updated prompt each retry.
    for attempt, raw in enumerate(attempts[: max_retries + 1]):
        data, err = validate(raw)
        if data is not None:
            return {"result": data, "attempts": attempt + 1, "model": model}
        prompt += f"\nYour previous output failed validation: {err}. Fix it."
        print(f"attempt {attempt + 1} failed: {err} -> re-asking")
    raise ValueError("validation failed after retries")


# Configure the framework object; this line prepares it but may not execute it yet.
model = route("classify_response")
print(f"routed classify_response -> {model}")
print(json.dumps(call_with_retry("Classify this response.", model), indent=2))


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
